## **Import libraries**

In [19]:
import random

In [20]:
# -Moises-
def fitness(chromosome: str) -> float:
    if not chromosome:
        return 0.0
    
    total_weight = 0
    for char in chromosome.lower():
        if 'a' <= char <= 'z':
            total_weight += ord(char) - ord('a') + 1
            
    max_possible_weight = 26 * len(chromosome)
    
    return total_weight / max_possible_weight

In [21]:
fitness("SKQ")

0.6025641025641025

## **Functions**

In [22]:
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

### **Initial Population**

In [23]:
def generate_initial_population(peptide_length, population_size):
    population = []
    
    for _ in range(population_size):
        peptide = ''.join(random.choices(amino_acids, k=peptide_length))
        population.append(peptide)
        
    return population

In [24]:
# Example
generate_initial_population(5, 4)

['GLRTV', 'YHFNH', 'TPQHS', 'AMMSG']

### **Crossover**

In [25]:
import random

def uniform_crossover(parent1, parent2):
    child = []
    
    for p1_gene, p2_gene in zip(parent1, parent2):
        # Randomly generate a binary mask value (0 or 1)
        # to determine which parent's gene will be selected.
        mask = random.randint(0, 1)
        
        if mask == 0:
            child.append(p1_gene)
        else:
            child.append(p2_gene)
            
    return "".join(child)

In [26]:
# Example
uniform_crossover("MTASGKLRVPHA", "TIHWSGATGAGA")

'MTHWGGATVAHA'

### **Mutation**

In [27]:
def mutate_sequence(sequence, mutation_rate_per_gene=0.8):
    mutated_sequence = []
    
    for gene in sequence:
        if random.random() < mutation_rate_per_gene:
            possible_mutations = [aa for aa in amino_acids if aa != gene]
           
            mutated_sequence.append(random.choice(possible_mutations))
        else:
            mutated_sequence.append(gene)
        #print(mutated_sequence)
    return "".join(mutated_sequence)

In [28]:
# Example
mutate_sequence("TISGHAW", mutation_rate_per_gene=0.8)

'NWSKMYH'

### **Binary Tournament**

In [29]:
import random

def tournament(population, tournament_prob=0.8):
    contenders = random.sample(population, 2)
    #print(contenders)
    best = max(contenders, key=lambda individual: individual['fitness'])

    if random.random() < tournament_prob:
        winner = best
    else:
        winner = random.choice(contenders)

    return winner

In [30]:
# Example
population = [
    {"sequence": "EKTTNMF", "fitness": 0.92},
    {"sequence": "HKVMRAT", "fitness": 0.31},
    {"sequence": "MMQYQEY", "fitness": 0.45},
    {"sequence": "MICRHPM", "fitness": 0.99},
    {"sequence": "CQRRCEW", "fitness": 0.12},
    {"sequence": "NQTSCSL", "fitness": 0.92},
    {"sequence": "TKHNYCN", "fitness": 0.03},
    {"sequence": "RDVEKFA", "fitness": 0.23}
]

tournament(population, tournament_prob=0.5)


{'sequence': 'HKVMRAT', 'fitness': 0.31}

## **Genetic Algorithm**

In [35]:
def genetic_algorithm(
    peptide_length,
    fitness_function,
    mutation_rate=0.8,
    mutation_rate_per_gene = 0.5,
    tournament_prob=0.8,
    generations=300,
    children_count=30,
):
    
    # Generate an initial population 
    raw_population = generate_initial_population(peptide_length, children_count)

    # Evaluate fitness for each individual in the initial population
    population = [
        {'sequence': seq, 'fitness': fitness_function(seq)}
        for seq in raw_population
    ]

    best_overall = max(population, key=lambda ind: ind['fitness'])

    # Evolutionary loop 
    for generation in range(generations):
        next_generation = []

        # Create a new generation of offspring
        for _ in range(children_count):

            # Select two parent individuals using tournament selection
            parent1 = tournament(population, tournament_prob)
            parent2 = tournament(population, tournament_prob)
            

            # Perform uniform crossover between selected parents
            child_sequence = uniform_crossover(
                parent1['sequence'],
                parent2['sequence']
            )

            # Decide whether this child mutates or not
            if random.random() < mutation_rate:
            # Apply mutation 
                child_sequence = mutate_sequence(child_sequence, mutation_rate_per_gene)

            # Evaluate fitness of the resulting offspring
            child_fitness = fitness_function(child_sequence)

            # Store the new individual in the next generation
            next_generation.append({
                'sequence': child_sequence,
                'fitness': child_fitness
            })

        # Replace the current population with the newly generated one
        population = next_generation

        # best of current generation
        current_best = max(population, key=lambda ind: ind['fitness'])

        # update global best
        if best_overall is None or current_best['fitness'] > best_overall['fitness']:
            best_overall = current_best

        # Identify and display the best individual of the current generation
        best_individual = max(population, key=lambda ind: ind['fitness'])
        print(
            f"Best Sequence: {best_individual['sequence']} | "
            f"Fitness: {best_individual['fitness']:.4f}"
        )

    return best_overall

In [36]:
genetic_algorithm(
    peptide_length=10,
    fitness_function = fitness,
    mutation_rate=0.8,
    mutation_rate_per_gene=0.3,
    tournament_prob=0.8,
    generations=300,
    children_count=30,
    )

Best Sequence: LTHTTEWKSY | Fitness: 0.6269
Best Sequence: LQHTTVWKSY | Fitness: 0.6808
Best Sequence: WQPTMPWYWQ | Fitness: 0.7423
Best Sequence: TTSTPPRWWV | Fitness: 0.7577
Best Sequence: TLPSPVWTPK | Fitness: 0.6731
Best Sequence: TLPSPVWTPK | Fitness: 0.6731
Best Sequence: VYKSPVWTPK | Fitness: 0.7115
Best Sequence: LRPSWTWTWQ | Fitness: 0.7346
Best Sequence: VRPTSTWWWM | Fitness: 0.7577
Best Sequence: YRSQWTWWWM | Fitness: 0.7846
Best Sequence: YYGVWTYWWY | Fitness: 0.8385
Best Sequence: YYLTWSYWQQ | Fitness: 0.7923
Best Sequence: PYCLWVYVWY | Fitness: 0.7538
Best Sequence: YVKMYYYVTR | Fitness: 0.7923
Best Sequence: WSMYWVTSWM | Fitness: 0.7692
Best Sequence: RYKWFYYVQQ | Fitness: 0.7269
Best Sequence: YNWRWWYHQN | Fitness: 0.7308
Best Sequence: IAYRWSYHYV | Fitness: 0.6731
Best Sequence: YCYRYNWWQG | Fitness: 0.6923
Best Sequence: YYYFWLYWEW | Fitness: 0.7385
Best Sequence: NTYYSELWWW | Fitness: 0.7269
Best Sequence: SMYIKRYWWW | Fitness: 0.7269
Best Sequence: SPYIKKYWWY | Fitn

{'sequence': 'YYYTSYWYTY', 'fitness': 0.8923076923076924}

# **Your turn!!**

In [33]:
## Function Mutation with restrictions; amino acids with same polarity

In [34]:
## Genetic with mu+lambda